# 02 — Build Index
Chunks `scraped_corpus.json` using every strategy, tags metadata, embeds, and upserts to Qdrant.
Run once per chunking strategy. Results land in separate Qdrant collections:
`hr_rag_recursive`, `hr_rag_semantic`, `hr_rag_sentence_window`, `hr_rag_parent_child`, `hr_rag_proposition`.

**Re-running is safe** — chunk IDs are deterministic (content-hash based), so unchanged chunks are
replaced in-place and new chunks are inserted. To fully wipe and rebuild a collection set `recreate=True`.

In [1]:
# ── Environment & path setup ─────────────────────────────────────────────────
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("QDRANT_URL       :", QDRANT_URL)

OPENAI_API_KEY set: True
QDRANT_URL       : https://38718a4e-130d-4ef7-be90-0c1893444e4c.us-east4-0.gcp.cloud.qdrant.io


In [2]:
# ── Clients ──────────────────────────────────────────────────────────────────
from openai import OpenAI
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding

oai        = OpenAI(api_key=OPENAI_API_KEY)
qdrant     = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
print("Clients ready.")

Clients ready.


In [3]:
# ── 4. Load corpus ────────────────────────────────────────────────────────────
import json
import config

with open(config.CORPUS_PATH, encoding='utf-8') as f:
    corpus = json.load(f)

print(f'Corpus: {len(corpus)} documents')

Corpus: 278 documents


In [ ]:
# ── 5. Chunk + index — run one cell per strategy ──────────────────────────────
# Change STRATEGY to: 'recursive', 'semantic', 'sentence_window', 'parent_child', 'proposition'

from pipeline.chunkers import get_chunks
from pipeline.embedder import build_index
import os

os.makedirs(config.CLEAN_CHUNKS_DIR, exist_ok=True)
os.makedirs(config.EMBEDDINGS_DIR,   exist_ok=True)

STRATEGY = 'proposition'   # ← change this

print(f'Chunking with strategy: {STRATEGY}')
chunks = get_chunks(STRATEGY, corpus, openai_api_key=OPENAI_API_KEY)
print(f'Chunks produced: {len(chunks):,}')

# Save chunks to Drive
chunks_path = f'{config.CLEAN_CHUNKS_DIR}/{STRATEGY}.json'
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f'Saved chunks to {chunks_path}')

# Tag, embed, upsert
build_index(
    chunks=chunks,
    strategy=STRATEGY,
    oai=oai,
    qdrant=qdrant,
    bm25_model=bm25_model,
    recreate=False,   # set True to wipe and rebuild the collection
)

Chunking with strategy: proposition


In [ ]:
# ── 6. Verify ─────────────────────────────────────────────────────────────────
info = qdrant.get_collection(f'{config.COLLECTION_PREFIX}_{STRATEGY}')
print(f'Collection: hr_rag_{STRATEGY} — {info.points_count:,} points')